# Activation Functions and Max Pooling in CNNs

**Course:** IBM AI Engineering Professional Certificate  
**Module:** Deep Neural Networks with PyTorch  
**Author:** Jack Pumpuni Frimpong-Manso  
**Estimated Time:** 25 minutes

---

## Learning Objectives
After completing this lab you will be able to:
- Apply a ReLU activation function to a convolutional output (activation map)
- Explain the role and effect of ReLU non-linearity in feature detection
- Implement max pooling and explain how it achieves spatial downsampling
- Understand the effect of stride on max pooling output size

---

## Table of Contents
1. [Activation Functions](#1.-Activation-Functions)
2. [Max Pooling](#2.-Max-Pooling)
3. [Combining Conv → ReLU → MaxPool](#3.-Combining-Conv-→-ReLU-→-MaxPool)
4. [Summary](#4.-Summary)

## Install and Import Libraries

In [ ]:
%%time
%pip install pandas numpy matplotlib scipy --quiet
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu --quiet

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, misc

print(f"PyTorch version: {torch.__version__}")
print("All libraries imported successfully.")

---
## 1. Activation Functions

### Why do we need activation functions?

A convolution is a **linear** operation. Stacking multiple linear operations gives you another linear operation — no matter how many layers deep, the network could be collapsed into a single layer with no gain in expressive power.

**Activation functions introduce non-linearity**, allowing the network to learn complex, hierarchical patterns.

### ReLU (Rectified Linear Unit)

$$\text{ReLU}(x) = \max(0, x)$$

- All negative values → 0
- Positive values → unchanged
- **Fast to compute**, avoids vanishing gradient (vs. sigmoid/tanh)
- Applied **element-wise** to every value in the activation map

In [ ]:
# Build the convolution layer with a Sobel-X kernel
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)
Gx = torch.tensor([[1.0, 0, -1.0],
                    [2.0, 0, -2.0],
                    [1.0, 0, -1.0]])
conv.state_dict()['weight'][0][0] = Gx
conv.state_dict()['bias'][0] = 0.0

print("Sobel-X kernel:")
print(conv.state_dict()['weight'][0][0])

In [ ]:
# Create 5×5 image with a vertical edge
image = torch.zeros(1, 1, 5, 5)
image[0, 0, :, 2] = 1
print("Input image:")
print(image[0, 0])

In [ ]:
# Step 1: Apply convolution
Z = conv(image)
print("Activation map Z (after convolution):")
print(Z[0, 0].detach())
print("\nNotice: negative values present (response to right side of edge)")

In [ ]:
# Step 2: Apply ReLU
A = torch.relu(Z)
print("After ReLU (negative values zeroed out):")
print(A[0, 0].detach())

# Also using the nn.ReLU module (equivalent)
relu = nn.ReLU()
A_module = relu(Z)
print("\nUsing nn.ReLU() module (same result):")
print(A_module[0, 0].detach())

In [ ]:
# Visualise ReLU effect on activation map
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

Z_np = Z[0, 0].detach().numpy()
A_np = A[0, 0].detach().numpy()

im0 = axes[0].imshow(image[0, 0].numpy(), cmap='gray')
axes[0].set_title('Input Image (5×5)\nVertical edge at col 2', fontsize=11)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(Z_np, cmap='RdBu', vmin=-4, vmax=4)
axes[1].set_title('Activation Map Z\n(after Conv, before ReLU)', fontsize=11)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{Z_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(A_np, cmap='Greens')
axes[2].set_title('After ReLU: max(0, Z)\n(negatives → 0)', fontsize=11)
for i in range(3):
    for j in range(3):
        color = 'black' if A_np[i,j] > 0 else 'red'
        axes[2].text(j, i, f'{A_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold', color=color)
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Convolution → Activation Map → ReLU', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('relu_activation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: relu_activation.png")

In [ ]:
# Compare activation functions: ReLU vs Sigmoid vs Tanh
x_vals = torch.linspace(-4, 4, 200)

relu_out   = torch.relu(x_vals)
sigmoid_out = torch.sigmoid(x_vals)
tanh_out   = torch.tanh(x_vals)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(x_vals.numpy(), relu_out.numpy(), 'b-', linewidth=2)
axes[0].set_title('ReLU: max(0, x)', fontsize=12)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_vals.numpy(), sigmoid_out.numpy(), 'r-', linewidth=2)
axes[1].set_title('Sigmoid: 1/(1+e⁻ˣ)', fontsize=12)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('x')
axes[1].grid(True, alpha=0.3)

axes[2].plot(x_vals.numpy(), tanh_out.numpy(), 'g-', linewidth=2)
axes[2].set_title('Tanh: (eˣ - e⁻ˣ)/(eˣ + e⁻ˣ)', fontsize=12)
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('x')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Common Activation Functions in CNNs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('activation_functions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: activation_functions_comparison.png")

---
## 2. Max Pooling

### What is Max Pooling?

Max pooling is a **spatial downsampling** operation that:
- Divides the feature map into non-overlapping (or overlapping) windows
- Retains only the **maximum value** from each window
- **Reduces spatial dimensions** → fewer parameters, less computation
- Provides **translation invariance**: a feature detected slightly off-position still activates

The output size formula is the same as convolution:
$$M_{new} = \left\lfloor \frac{M - K_{pool}}{s} \right\rfloor + 1$$

In [ ]:
# Create a 4×4 image with known values
image1 = torch.zeros(1, 1, 4, 4)
image1[0, 0, 0, :] = torch.tensor([1.0, 2.0, 3.0, -4.0])
image1[0, 0, 1, :] = torch.tensor([0.0, 2.0, -3.0, 0.0])
image1[0, 0, 2, :] = torch.tensor([0.0, 2.0, 3.0, 1.0])
image1[0, 0, 3, :] = torch.tensor([0.0, 1.0, 1.0, 0.0])

print("Input feature map (4×4):")
print(image1[0, 0])

In [ ]:
# Max Pooling with stride=1 (overlapping windows)
max1 = torch.nn.MaxPool2d(2, stride=1)
result1 = max1(image1)
print("Max Pooling (kernel=2, stride=1) — overlapping windows:")
print(result1[0, 0].detach())
print(f"Output shape: {result1.shape[2:]}")
print(f"  → M_new = floor((4-2)/1) + 1 = 3")

In [ ]:
# Max Pooling with stride=None (defaults to kernel size — non-overlapping)
max2 = torch.nn.MaxPool2d(2)  # stride defaults to kernel_size=2
result2 = max2(image1)
print("Max Pooling (kernel=2, stride=2) — non-overlapping windows:")
print(result2[0, 0].detach())
print(f"Output shape: {result2.shape[2:]}")
print(f"  → M_new = floor((4-2)/2) + 1 = 2")
print()
print("Explanation of each output value:")
print(f"  Top-left 2×2 window:     max(1,2,0,2)  = {max(1,2,0,2)}")
print(f"  Top-right 2×2 window:    max(3,-4,-3,0) = {max(3,-4,-3,0)}")
print(f"  Bottom-left 2×2 window:  max(0,2,0,1)  = {max(0,2,0,1)}")
print(f"  Bottom-right 2×2 window: max(3,1,1,0)  = {max(3,1,1,0)}")

In [ ]:
# Visualise Max Pooling step by step
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Original
img_np = image1[0, 0].numpy()
im0 = axes[0].imshow(img_np, cmap='coolwarm', vmin=-4, vmax=4)
axes[0].set_title('Input Feature Map (4×4)', fontsize=11)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{img_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im0, ax=axes[0])

# Stride=1
r1_np = result1[0, 0].detach().numpy()
im1 = axes[1].imshow(r1_np, cmap='coolwarm')
axes[1].set_title('MaxPool(2, stride=1)\n→ 3×3 output', fontsize=11)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{r1_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=axes[1])

# Stride=2
r2_np = result2[0, 0].detach().numpy()
im2 = axes[2].imshow(r2_np, cmap='coolwarm')
axes[2].set_title('MaxPool(2, stride=2)\n→ 2×2 output (non-overlapping)', fontsize=11)
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f'{r2_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Max Pooling: Spatial Downsampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('max_pooling_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: max_pooling_demo.png")

---
## 3. Combining Conv → ReLU → MaxPool

In a real CNN, these three operations form the **core building block** of each convolutional layer:

```
Input → Conv2d → ReLU → MaxPool2d → (next layer)
```

- **Conv2d:** Detects features (edges, textures, shapes)
- **ReLU:** Introduces non-linearity, suppresses weak responses
- **MaxPool2d:** Reduces spatial size, retains dominant features, provides invariance

In [ ]:
# Build a mini pipeline: Conv → ReLU → MaxPool
torch.manual_seed(0)

# Define a 7×7 test image with a central vertical edge
image_full = torch.zeros(1, 1, 7, 7)
image_full[0, 0, :, 3] = 1.0  # vertical edge at column 3

# Conv layer with Sobel-X
conv_layer = nn.Conv2d(1, 1, kernel_size=3, padding=0)
conv_layer.state_dict()['weight'][0][0] = torch.tensor([[ 1.0,  0.0, -1.0],
                                                         [ 2.0,  0.0, -2.0],
                                                         [ 1.0,  0.0, -1.0]])
conv_layer.state_dict()['bias'][0] = 0.0

# ReLU
relu_layer = nn.ReLU()

# MaxPool
pool_layer = nn.MaxPool2d(2, stride=2)

# Forward pass
Z = conv_layer(image_full)     # Conv: 7→5
A = relu_layer(Z)              # ReLU: shape unchanged
P = pool_layer(A)              # Pool: 5→2

print(f"Input shape:   {image_full.shape}")
print(f"After Conv:    {Z.shape}  (7-3+1=5)")
print(f"After ReLU:    {A.shape}  (shape unchanged)")
print(f"After MaxPool: {P.shape}  (floor((5-2)/2)+1=2)")

print("\nConv output Z:")
print(Z[0, 0].detach())
print("\nAfter ReLU A:")
print(A[0, 0].detach())
print("\nAfter MaxPool P:")
print(P[0, 0].detach())

In [ ]:
# Full pipeline visualisation
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

stages = [
    (image_full[0, 0].numpy(), 'Input (7×7)', 'gray'),
    (Z[0, 0].detach().numpy(),  'Conv Output Z (5×5)', 'RdBu'),
    (A[0, 0].detach().numpy(),  'After ReLU A (5×5)', 'Greens'),
    (P[0, 0].detach().numpy(),  'After MaxPool P (2×2)', 'YlOrRd'),
]

for ax, (data, title, cmap) in zip(axes, stages):
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title, fontsize=10)
    H, W = data.shape
    for i in range(H):
        for j in range(W):
            ax.text(j, i, f'{data[i,j]:.0f}',
                    ha='center', va='center', fontsize=10,
                    color='black' if abs(data[i,j]) < 3 else 'white')
    plt.colorbar(im, ax=ax)

plt.suptitle('CNN Building Block: Conv → ReLU → MaxPool', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_pipeline_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: cnn_pipeline_demo.png")

---
## 4. Summary

| Component | Operation | Purpose |
|---|---|---|
| `nn.Conv2d` | Element-wise multiply + sum, sliding kernel | Feature detection (edges, textures) |
| `nn.ReLU` | $\max(0, x)$ applied element-wise | Introduces non-linearity, sparsity |
| `nn.MaxPool2d` | Max value in each pooling window | Downsampling, translation invariance |

### Key formulas recap

| Operation | Output Size |
|---|---|
| Conv (basic) | $M - K + 1$ |
| Conv (stride s, padding p) | $\lfloor(M + 2p - K)/s\rfloor + 1$ |
| MaxPool (kernel K, stride s) | $\lfloor(M - K)/s\rfloor + 1$ |

In [ ]:
# Dimension tracking through a typical CNN block
print("Dimension tracking: Conv → ReLU → MaxPool")
print("=" * 50)

M = 32  # e.g., 32×32 input
K_conv = 5
p_conv = 2  # 'same' padding for k=5
s_conv = 1
K_pool = 2
s_pool = 2

after_conv = (M + 2*p_conv - K_conv) // s_conv + 1
after_relu = after_conv  # shape unchanged
after_pool = (after_relu - K_pool) // s_pool + 1

print(f"Input:          {M}×{M}")
print(f"Conv2d(K={K_conv}, s={s_conv}, p={p_conv}):   {after_conv}×{after_conv}")
print(f"ReLU:           {after_relu}×{after_relu}  (unchanged)")
print(f"MaxPool(K={K_pool}, s={s_pool}):  {after_pool}×{after_pool}")
print(f"\nDownsampling ratio: {M}/{after_pool} = {M/after_pool:.1f}x")